In [2]:
import numpy as np

In [4]:
np.zeros(3)   #0이 3개 들어있는 1차원 배열

array([0., 0., 0.])

In [3]:
np.zeros((0, 2))  #행(가로)은 0개이고, 열(세로)은 2개인 2차원 배열
# 나중에 데이터가 들어오면 무조건 2개씩 짝지어서(예: [반복횟수, 손실값]) 아래로 차곡차곡 쌓을 거야"**라고
# **틀(Shape)**만 미리 잡아두는 것

array([], shape=(0, 2), dtype=float64)

### bias-variance trade-off

bias와 variance는 trade-off(상충 관계)
```
bias가 크면 -> 과소적합underfitting
variance가 크면 -> 과대적합overfitting
```

bias-variance trade-off로 인한 문제를 해결하기 위해서는, 적당한 중간값으로 최적화시켜야 한다.
```
1. 정규화regulation : weight 크기가 너무 커지거나 작게 하지 않도록 한다.
2. 드롭아웃dropout : 은닉층의 몇몇 뉴런들을 랜덤으로 일부 비활성화시킴으로써 overfitting을 방지한다.
3. 데이터 증강data augumentation : 데이터를 다양하게 만들어 overfitting 방지한다.
4. 조기종료early stopping : 검증 성능이 나빠지기 전에 중지시킨다.
```

###train, val, test셋 비율
```
1. 대용량 데이터 : 8/1/1
2. 중소규모(1만개 이하) 데이터 : 6/2/2
3. 소량 데이터(1천개 이하) : K-Fold 교차검증 사용

###모델 평가 지표
```
1. 회귀 : MAE / RMSE, R**2
2. 균형 분류 : Accuracy / F1-score
3. 불균형 분류 : F1-score / Precision, Recall
4. 의료 진단 : Recall / F1-score
5. 스팸 필터 : Precision / F1-score
```


ROC-AUC ??

###신경망 그림의 '층(Layer)'은 데이터(텐서)를, 파이토치에서 말하는 '레이어 함수'는 그 데이터를 가공하는 함수를 의미하는 전혀 다른 대상입니다.
```
1. 레이어 함수: nn.Linear, nn.ReLU 텐서 출력 및 모델 부품
2. 파라미터: w, b 레이어 함수 내부에 저장되어 학습을 통해 값 조정됨
```

In [ ]:
# nn.Sequential을 사용해 전체를 하나의 합성 함수로 정의
net2 = nn.Sequential(
l1,
relu,
l2
)
# 5. 한 번에 결과 계산
outputs2 = net2(inputs)

#### 활성화 함수의 의미
비선형성을 추가->인공지능 모델이 실제의 데이터를 예측할 수 있도록 함. 비선형성 없으면? 인공지능 필요없지... 선형 예측만 할 수 있으니까

####학습의 4단계 순환
```
1. 예측 계산 outputs=net(inputs)
2. 손실 계산 criterion(outputs, labels)
3. 경사 계산 loss.backward()
4. 파라미터 수정 optimizer.step()

* optimizer = with torch.no_grad(): 가중치 업데이트 항
```

####Autograd Hook : pytorch에서 역전파 과정 중 특정 텐서의 gradient를 가로채서 확인/수정할 수 있는 기능
why? 중간에 gradient가 소실/폭발하는 거 관찰해서 디버깅하는 등의 용도로 쓴다.

####학습률 스케줄러lr scheduler : w -= lr * grad

이상적인 lr : 초반에는 대략적으로 큰 폭, 나중에는 세밀하게 작은 폭
```
1. StepLR : 일정 epoch마다 lr을 고정 비율로 감소
2. CosineAnnealingLR : cos 곡선을 따라 lr이 초반에는 빠르게, 후반에는 매우 천천히 감소 (이미지 분류에 좋음)
3. OneCycleLR(최신) : lr을 증가시켰다가 다시 감소시키는 전략. Warm-up으로 안정성 확보하고, 높은 학습률로 빠르게 학습한 뒤, Decay로 정밀하게 마무리. (대규모 학습에 최적)
```

####가중치 초기화 문제
```
1. 모든 가중치=0: 모든 뉴런이 똑같이 학습되는 대칭성 문제(3개가 있어도 1개처럼 작동함)
2. 가중치 너무 크면: 활성화 함수 터지고 gredient 소실됨
3. 가중치 너무 작으면: 신호가 너무 작아서 전달이 안됨
```
####해결: Xaveir VS He 초기화
```
보통은 He 초기화 사용(CNN+ReLU, BN 사용시) 추천
1. Xavier(2010) : 입출력 뉴런 개수를 고려해서 가중치 분산을 조절한다. 대칭 활성화 함수(Tanh, Sigmoid)에 최적화된 방법, 양방향 신호 흐름을 설계.
2. He(2015) : ReLU에 적합. ReLU가 음수를 0으로 만들어 절반의 뉴런이 비활성화되는 특성을 고려함. 더 큰 초기 가중치로 신호를 보존하고, 일방통행 신호 흐름을 설계한다.

####Full Batch VS Mini Batch

작은 배치 사이즈(16-32)
장점:
- 정규화 효과 (노이즈가 많아 일반화 좋음)
- 메모리 절약
단점:
- 불안정한 학습 (그래디언트 노이즈)
- 느린 계산 (GPU 활용도 낮음)

큰 배치 사이즈(256-1024)
장점:
- 안정적 학습 (정확한 그래디언트)
- 빠른 계산 (GPU 병렬화)
단점:
- 일반화 성능 저하
- 메모리 부족 위험

####BN (batch normalization) : 평균0 분산1

모델의 층과 층 사이에서 데이터 값이 너무 들쭉날쭉하지 않게, 배치(묶음) 단위로 평균과 분산을 구해서 데이터를 예쁘게 다듬어(정규화해서) 다음 층으로 넘겨주는 것
```
딥러닝 모델은 수많은 층(Layer)이 이어달리기를 하는 것과 같습니다.

1. 상황 (BN 없음):

**1번 층(Layer 1)**이 데이터를 처리해서 2번 층으로 넘겨줍니다.

그런데 어떤 배치(데이터 묶음)에서는 1번 층이 보낸 값이 **엄청 큰 숫자(예: 100, 500)**들입니다.

다음 배치에서는 **엄청 작은 숫자(예: 0.01, 0.02)**들입니다.

2번 층의 반응: "아니, 들어오는 숫자가 매번 너무 달라서(분포가 달라서) 어느 장단에 맞춰 춤을 춰야(학습해야) 할지 모르겠네!" → 학습이 잘 안됨 (내부 공변량 변화, Internal Covariate Shift).

2. 상황 (BN 있음):

1번 층과 2번 층 사이에 **BN(반장님)**이 서 있습니다.

BN의 역할: 1번 층에서 넘어온 데이터 한 묶음(배치)을 봅니다. "어? 이번엔 평균이 500이네? 너무 커. 평균 0, 분산 1로 깎아서 2번 층한테 넘겨줘."

2번 층의 반응: "오, 들어오는 데이터가 항상 0 근처로 일정하네? 이제 맘 편히 학습에만 집중하면 되겠다!" → 학습이 엄청 잘됨 (안정성 확보).
```

* layer normalization : 각 샘플의 특징 차원에 대해 정규화 RNN, transformer에 적합 (RNN은 시퀀스 길이 달라서 BN 별로)
* Group Normalization : ResNet에 적합. 채널을 그룹으로 나눠 정규화.

####손실 함수
손실 함수 = 모델의 예측값과 실제값의 차이를 계산하는 함수

* 회귀 손실 함수
```
1. MSE (mean squared error) : 직관적. 이상치에 민감
2. MAE : 절댓값으로 계산해서 이상치에 덜 민감, 0 근처에서 미분 불가능->최적화 힘듦.
3. Huber Loss : 작은 오차에는 MSE, 큰 오차에는 MAE로 전환하는 손실함수
```

* 이진 분류 : BCELoss (0,1), BCEWithLogitsLoss (Sigmoid+BCELoss교차 엔트로피 -> 모델의 출력층에 sigmoid 굳이 넣지 않아도 되고, 계산 과정의 수치 불안정성을 해결함)

* 다중 분류 : CrossEntropyLoss() (입력 형식 주의, 점수 -> 확률(Softmax) -> 정답과 비교(One-hot)를 한번에 처리하는 함수)
```
# 1. 예측값 (predictions): [배치 크기, 클래스 개수]
# 모델: "내 생각엔 0번은 2.0점, 1번은 1.0점, 2번은 0.1점 같아."
predictions = tensor([[2.0, 1.0, 0.1]])

# 2. 정답 (target): [배치 크기]
# 정답지: "정답은 0번이야." (원-핫 인코딩인 [1, 0, 0]을 쓸 필요 없음!)
target = tensor([0])

# 3. 채점 (loss 계산)
# 파이토치: "오케이, 네가 0번에 제일 높은 점수(2.0) 줬네? 정답도 0번이고. 잘했어, 오차(Loss)는 작게 줄게."
loss = CrossEntropyLoss()(predictions, target)
```

####손실 곡면 = 가중치 공간에서 손실값의 변화를 나타낸 지형 (높낮이 있음)

좋은 손실 곡면 만들기
```
1. BN, He/Xavier 초기화, minibatch
2. gradient cliping(폭발 방지), warm-up(초반 탐색 안정성), 적절한 lr, 손실값 모니터링(학습률 스케줄러, OneCycleLR->이상징후 조기발견)

In [ ]:
# 파이토치의 텐서는 기본적으로 배치 단위로 데이터를 처리하므로 2차원 텐서 입력을 해줘야 한다.
view(-1, N)

In [ ]:
# 입력: 2, 출력: 3 선형 함수 정의
l3 = nn.Linear(2, 3)

# 초깃값 설정 (weight는 각 행별로 다르게 설정하고, bias는 모두 2.0으로 고정)
nn.init.constant_(l3.weight[0, :], 1.0)
nn.init.constant_(l3.weight[1, :], 2.0)
nn.init.constant_(l3.weight[2, :], 3.0)
nn.init.constant_(l3.bias, 2.0)

# y3의 data:
# tensor([[2., 2., 2.],
# [3., 4., 5.],
# [3., 4., 5.],
# [4., 6., 8.]])

In [ ]:
class Net(nn.Module): # 모든 모델의 부모 클래스인 nn.Module:파이토치의 강력한 기능들 모두 상속
  def __init__(self, n_input, n_output):
    super().__init__()    # == super(nn.Module).__init__()
    # 출력층 정의: 입력 특성 수(n_input)와 출력 특성 수(n_output)를 받는 선형 레이어(l1)를 정의합니다.
    self.l1 = nn.Linear(n_input, n_output)

  # 예측 함수 정의: 데이터의 순전파 흐름을 정의합니다.
  def forward(self, x):
    x1 = self.l1(x)
    return x1

In [ ]:
# 학습 루프 구현

# 학습률
lr = 0.01
# 인스턴스 생성 (학습 시작 전 파라미터 값 초기화)
net = Net(n_input, n_output)
criterion = nn.MSELoss() # 손실 함수: 평균 제곱 오차
optimizer = optim.SGD(net.parameters(), lr=lr) # 최적화 함수: 경사 하강법
num_epochs = 5000 # 반복 횟수
# 평가 결과 기록 (손실 값만 기록)
history = np.zeros((0, 2))
# 반복 계산 메인 루프
for epoch in range(num_epochs):
  optimizer.zero_grad()
  # ① 경삿값 초기화 (가장 먼저 수행!)
  outputs = net(inputs)
  # ② 예측 계산
  loss = criterion(outputs, labels)
  loss.backward()# ④ 경사 계산
  optimizer.step()# ⑤ 파라미터 수정

####net(inputs) 호출의 비밀 : __ call __
net : 정의한 모델 클래스 Net()의 인스턴스
Net() 내부에 forward라는 함수가 정의되어 있는데, 실제 모델 예측 순전파 코드는 net.forward(x)가 아닌 net(x)를 사용한다.

why?
파이썬의 __call__ 기능 덕분에 net(x)라고 쓰면 알아서 **net.forward(x)** 를 실행해줍니다. 그런데 그냥 실행하는 게 아니라 앞뒤로 중요한 관리 작업을 해주기 때문에, 반드시 net(x)라고 써야 합니다.


#### 이진 로지스틱 회귀 모델 구조
로지스틱 회귀: 분류 문제를 풀기 위한 알고리즘. 하나 이상의 독립변수로 어떤 사건이 발생할 확률을 예측하는 모델. 일반 회귀모델 아니고 분류모델임.
```
입력 텐서 -> 선형 함수(nn.Linear) -> nn.Sigmoid -> 출력 텐서

* 이진 로지스틱 회귀 모델 = 선형 회귀 모델 - sigmoid
* 손실함수 : 교차 엔트로피 함수(모델의 confidence가 높으면 더 큰 패널티 부여)
* 결정 경계(데이터 공간을 가르는 하나의 직선) 학습. 일반 회귀모델은 결정경계 없다.
```

#### 회귀Regression VS 분류Classification
```
1. 회귀(Regression): 데이터들이 흩뿌려져 있을 때, 그 점들을 가장 잘 대변하는(가장 가깝게 지나가는) 하나의 선을 긋는 것, 입력이 변할 때 결과가 어떻게 변하는지 그 '비례 관계(기울기)'를 학습
2. 분류(Classification): 데이터들이 두 그룹(예: 개와 고양이)으로 섞여 있을 때, 두 그룹을 명확하게 갈라놓는 선을 긋는 것, 이 선을 기준으로 "넘어가면 A, 안 넘어가면 B"라고 판단할 수 있는 경계선을 학습
```

